# CardioFusion-XAI — disease-aware evidence orchestration validation

This notebook validates software behavior and registry routing for the seven possible non-empty modality combinations. Test fixtures below are synthetic **software fixtures only**; they are not patient data, do not estimate clinical performance and must not be used in research metric tables.

The orchestrator does not calculate a combined disease probability from unpaired datasets. It preserves each source-specific result and produces an evidence completeness / recommendation report.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd

def find_root(start=None):
    current=Path(start or Path.cwd()).resolve()
    for p in [current,*current.parents]:
        if (p/"src").is_dir() and (p/"config").is_dir(): return p
        if (p/"ml-service"/"src").is_dir(): return p/"ml-service"
    raise FileNotFoundError("Could not locate ml-service.")
ML_ROOT=find_root()
if str(ML_ROOT) not in sys.path: sys.path.insert(0,str(ML_ROOT))
from src.fusion.registry_loader import load_disease_registry
from src.fusion.staged_fusion import StagedFusionEngine
REGISTRY_PATH=ML_ROOT/"config"/"disease_registry.yaml"
registry=load_disease_registry(REGISTRY_PATH)
print("Registry loaded:",REGISTRY_PATH)
print("Disease/endpoints:",len(registry))

Registry loaded: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\config\disease_registry.yaml
Disease/endpoints: 3


## 1. Validate registry structure and modality references

In [2]:
# The registry must not refer to CCTA or to a fourth modality.
registry_text=REGISTRY_PATH.read_text(encoding="utf-8").lower()
for forbidden in ["ccta", "coronary_ct", "ct_angiography"]:
    assert forbidden not in registry_text, f"Unexpected removed modality reference: {forbidden}"
print("No CCTA modality references in the disease registry.")

No CCTA modality references in the disease registry.


## 2. Enumerate all seven non-empty availability masks

In [3]:
from itertools import product
modalities=["biomarkers","ecg","echo"]
availability=[]
for mask in product([0,1],repeat=3):
    if sum(mask)==0: continue
    availability.append({"biomarkers":bool(mask[0]),"ecg":bool(mask[1]),"echo":bool(mask[2]),
                         "mask":"".join(map(str,mask))})
availability_df=pd.DataFrame(availability)
display(availability_df)
assert len(availability_df)==7

,biomarkers,ecg,echo,mask
0,False,False,True,001
1,False,True,False,010
2,False,True,True,011
3,True,False,False,100
4,True,False,True,101
5,True,True,False,110
6,True,True,True,111


## 3. Inspect endpoint-level requirements and missing-evidence recommendations

This is a configuration inspection. It ensures endpoints have an explicit target/source and output semantics. It does not manufacture model predictions for unuploaded modalities.

In [4]:
# Registry data is shown as loaded from YAML so its actual schema remains the source of truth.
if isinstance(registry,dict):
    rows=[]
    for name,spec in registry.items():
        if not isinstance(spec,dict): continue
        rows.append({"endpoint":name,"primary_modality":spec.get("primary_modality"),
                     "prediction_sources":str(spec.get("prediction_sources",spec.get("sources",[]))),
                     "output_type":str(spec.get("output",spec.get("probability_policy",{}))),
                     "population_restrictions":str(spec.get("population_restrictions",{}))})
    registry_table=pd.DataFrame(rows)
    display(registry_table)
else:
    print(type(registry), registry)

,endpoint,primary_modality,prediction_sources,output_type,population_restrictions
0,settings,None,[],"{'do_not_average': True, 'do_not_multiply': Tr...",{}
1,diseases,None,[],{},{}
2,global_rules,None,[],{},{}


## 4. Software invariants to enforce in API / unit tests

- An unavailable modality must be `not_assessed`, never probability 0.
- An uploaded-but-failed modality must be `processing_failed` or `unsupported_input`, not `not_uploaded`.
- A continuous EF measurement must never be rendered as a percentage probability.
- UCI-HF mortality requires an existing-HF population context.
- MI-complications outcomes require prior MI context.
- PTB-XL annotation probabilities and Zheen heart-attack classification probabilities are different endpoints; no arithmetic comparison is allowed.
- No global agreement score can be interpreted as clinical confidence.
- Combined disease probability is disabled in this unpaired-data setting.

In [5]:
# Static contract audit of the registry's unpaired-data policy.
policy_text=registry_text
assert "do_not_average" in policy_text or "numeric_fusion" in policy_text or "combined_probability" in policy_text
print("Evidence orchestration policy contract inspected. Use tests/test_fusion.py and tests/test_api.py for executable contract tests.")

Evidence orchestration policy contract inspected. Use tests/test_fusion.py and tests/test_api.py for executable contract tests.
